In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import cv2


os.environ['CUDA_VISIBLE_DEVICES'] = ''

from ultralytics import YOLO

class SimpleTester:
    def __init__(self, model_name="best.pt", conf=0.3):
        print(f" Загрузка модели {model_name} (режим: CPU)...")
        self.model = YOLO(model_name)
        self.model.to('cpu')  
        self.conf = conf
        self.results = []
        
    def test_image(self, image_path, save_viz=True):
        """Протестировать одно изображение"""
        if not os.path.exists(image_path):
            return None
            
        print(f"\nАнализ: {Path(image_path).name}")
        
        results = self.model(image_path, conf=self.conf, device='cpu')
        result = results[0]
        
        if result.boxes is None or len(result.boxes) == 0:
            print(" Ничего не обнаружено")
            return {"image": Path(image_path).name, "detections": 0, "classes": {}}
        
        classes_found = {}
        for box in result.boxes:
            class_name = self.model.names[int(box.cls[0])]
            conf_val = float(box.conf[0])
            
            if class_name in ['car', 'bus', 'truck', 'wall']:  # Целевые классы
                if class_name not in classes_found:
                    classes_found[class_name] = []
                classes_found[class_name].append(conf_val)
                print(f" {class_name}: {conf_val:.3f}")
        
        if save_viz:
            os.makedirs("test_results", exist_ok=True)
            annotated = result.plot()
            output_path = f"test_results/annotated_{Path(image_path).name}"
            cv2.imwrite(output_path, annotated)
            print(f" Сохранено: {output_path}")
        
        return {
            "image": Path(image_path).name,
            "detections": sum(len(v) for v in classes_found.values()),
            "classes": classes_found
        }
    
    def test_folder(self, folder_path):
        """Протестировать все изображения в папке"""
        folder = Path(folder_path)
        
        # Ищем все изображения
        extensions = {'.jpg', '.jpeg', '.png', '.bmp'}
        images = [f for f in folder.iterdir() if f.suffix.lower() in extensions]
        
        if not images:
            print(f"В папке {folder_path} нет изображений")
            return []
        
        print(f"Найдено изображений: {len(images)}")
        print("=" * 50)
        
        for img in sorted(images):
            result = self.test_image(str(img))
            if result:
                self.results.append(result)
        
        self.print_summary()
        return self.results
    
    def print_summary(self):
        """Вывод статистики"""
        if not self.results:
            print("\nНет данных для анализа")
            return
        
        # Собираем всю статистику
        all_confs = {"car": [], "bus": [], "truck": [], "wall": []}
        
        for res in self.results:
            for cls, confs in res["classes"].items():
                if cls in all_confs:
                    all_confs[cls].extend(confs)
        
        print("\n" + "=" * 50)
        print("ИТОГОВАЯ СТАТИСТИКА")
        print("=" * 50)
        
        total_detections = 0
        all_conf_values = []
        
        for cls, confs in all_confs.items():
            if confs:
                avg_conf = np.mean(confs)
                total_detections += len(confs)
                all_conf_values.extend(confs)
                
                # Определяем статус
                if avg_conf >= 0.8:
                    status = " Отлично"
                elif avg_conf >= 0.65:
                    status = "Хорошо"
                elif avg_conf >= 0.5:
                    status = "Средне"
                else:
                    status = "Плохо"
                    
                print(f"{cls.capitalize():10} | Найдено: {len(confs):3} | "
                      f"Ср.уверенность: {avg_conf:.3f} | {status}")
        
        if all_conf_values:
            overall_avg = np.mean(all_conf_values)
            print(f"\nОбщая средняя уверенность: {overall_avg:.3f}")
            
            if overall_avg >= 0.7:
                print("Модель генерации работает ОТЛИЧНО!")
            elif overall_avg >= 0.55:
                print("Модель генерации работает ХОРОШО")
            else:
                print("Модель требует доработки")
        else:
            print("\nОбъекты не обнаружены")

TEST_FOLDER = "generated_images"  

tester = SimpleTester(model_name="best.pt", conf=0.3)

if os.path.exists(TEST_FOLDER):
    results = tester.test_folder(TEST_FOLDER)
else:
    print("Создайте папку и поместите туда сгенерированные изображения, или укажите другой путь в переменной TEST_FOLDER")